In [1]:
# Quarterly depedency graph

In [1]:
import os
os.getcwd()

'c:\\Users\\ugne.keliauskaite\\Bruegel GitLab\\research-2021-11-european-natural-gas-imports\\standalone pieces of code'

In [2]:
#Share_point = r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2021-11 European natural gas imports\Data' # Gio
Share_point = r'C:\Users\ugne.keliauskaite\Bruegel\Research - 2021-11 European natural gas imports\Data' # Ugne
os.chdir(Share_point)

In [3]:
import json
import requests
import pandas as pd
import numpy as np

from datetime import datetime
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.dates import DateFormatter
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns

In [4]:
# import ENTSOG pipeline data
entsog = pd.read_csv(r'Imports\EU27\df1.csv')
del entsog['dates.1']
entsog = entsog.set_index(pd.DatetimeIndex(entsog['dates']))
del entsog['dates']

In [20]:
# import LNG data from GIE (we only know where the LNG arrives, not where it comes from)
# agsi = pd.read_csv(r'C:\\Users\\giovanni.sgaravatti\\Bruegel\\Research - 2021-11 European natural gas imports\\Code\\LNG terminals\\raw_data\\agsi.csv') # Gio
agsi = pd.read_csv(r'C:\\Users\\ugne.keliauskaite\\Bruegel\\Research - 2021-11 European natural gas imports\\Code\\LNG terminals\\raw_data\\agsi.csv') # Ugne
agsi=agsi.set_index(pd.DatetimeIndex(agsi['dates']))
del agsi['dates']
del agsi['index']

In [22]:
# import Bloomberg LNG data (with these data we know both where it comes from and where it arrives, but we trust GIE better - also to be consistent with the tracker)
lng_b = pd.read_excel(r'LNG\Bloomberg\granular LNG imports.xlsx') # Gio
lng_b.rename(columns= {'Unnamed: 0':'dates'},inplace=True)
lng_b = lng_b.set_index(pd.DatetimeIndex(lng_b['dates']))
del lng_b['dates']

In [23]:
lng_b['Tot'] = lng_b.sum(axis=1,numeric_only=True)

In [24]:
# Divide each column by the 'Tot' column
ratios_df = lng_b.div(lng_b['Tot'], axis=0)

In [29]:
agsi_m = agsi.groupby(pd.Grouper(freq='M'))['sendOut'].sum(numeric_only=True)
# take only values after 2019 to be consistent with Bloomberg data
agsi_19 = agsi_m['2019':]

In [30]:
# to align with Bloombgerg lng
agsi_19= agsi_19.iloc[:-1] #Ugne change this one!

In [31]:
agsi_19

dates
2019-01-31     65357.1
2019-02-28     59014.4
2019-03-31     84049.6
2019-04-30     88014.5
2019-05-31     80457.8
2019-06-30     81049.8
2019-07-31     77001.5
2019-08-31     69694.7
2019-09-30     63490.6
2019-10-31     67550.0
2019-11-30     94085.4
2019-12-31     83621.2
2020-01-31     83265.8
2020-02-29     78679.3
2020-03-31     90039.7
2020-04-30     86774.6
2020-05-31     89312.7
2020-06-30     60129.1
2020-07-31     69732.6
2020-08-31     57500.1
2020-09-30     56064.0
2020-10-31     55343.8
2020-11-30     56292.5
2020-12-31     46510.4
2021-01-31     40463.0
2021-02-28     46866.3
2021-03-31     84268.6
2021-04-30     88432.4
2021-05-31     77472.8
2021-06-30     61439.9
2021-07-31     53765.9
2021-08-31     47731.8
2021-09-30     51209.3
2021-10-31     57892.7
2021-11-30     74198.5
2021-12-31     77652.1
2022-01-31    107437.5
2022-02-28     91405.1
2022-03-31    108125.1
2022-04-30    110093.5
2022-05-31    112455.2
2022-06-30    104995.3
2022-07-31    119804.0
2022-

In [32]:
# create new dataframe
lng = pd.DataFrame()
lng['dates'] = agsi_19.index

In [33]:
agsi_19

dates
2019-01-31     65357.1
2019-02-28     59014.4
2019-03-31     84049.6
2019-04-30     88014.5
2019-05-31     80457.8
2019-06-30     81049.8
2019-07-31     77001.5
2019-08-31     69694.7
2019-09-30     63490.6
2019-10-31     67550.0
2019-11-30     94085.4
2019-12-31     83621.2
2020-01-31     83265.8
2020-02-29     78679.3
2020-03-31     90039.7
2020-04-30     86774.6
2020-05-31     89312.7
2020-06-30     60129.1
2020-07-31     69732.6
2020-08-31     57500.1
2020-09-30     56064.0
2020-10-31     55343.8
2020-11-30     56292.5
2020-12-31     46510.4
2021-01-31     40463.0
2021-02-28     46866.3
2021-03-31     84268.6
2021-04-30     88432.4
2021-05-31     77472.8
2021-06-30     61439.9
2021-07-31     53765.9
2021-08-31     47731.8
2021-09-30     51209.3
2021-10-31     57892.7
2021-11-30     74198.5
2021-12-31     77652.1
2022-01-31    107437.5
2022-02-28     91405.1
2022-03-31    108125.1
2022-04-30    110093.5
2022-05-31    112455.2
2022-06-30    104995.3
2022-07-31    119804.0
2022-

In [34]:
 # multiply Bloomberg LNG ratios by AGSI totals
# and convert to M3m
for column in ratios_df.columns:
    lng[column] = agsi_19.values*ratios_df[column].values/10.3

In [35]:
lng.set_index(pd.DatetimeIndex(lng['dates']),inplace=True)
del lng['dates']

In [36]:
lng['Total less Russia and USA'] = lng['Tot'] - lng['Russia'] - lng['United States']

In [37]:
# Change the months for which you have data here
months = pd.date_range(start='2019-01-01', end='2023-12-31', freq='M')

In [38]:
entsog = entsog['2019':]

In [39]:
converter = 10300000   ## KWh to M3m --> 10.3 KWh/m^3     # on ENTSOG/AGSI the data comes in KWh, we transform it (later on) in M3m 

In [40]:
entsog_m = pd.DataFrame()
entsog_m['dates'] = months
entsog_m.set_index(pd.DatetimeIndex(entsog_m['dates']),inplace=True)
del entsog_m['dates']

for country in ['Russia', 'Norway','Algeria', 'UK', 'Azerbaijan','Libya']:
    entsog_m[country] = entsog[entsog['aggregation'] == country]['values'].groupby(pd.Grouper(freq='M')).sum(numeric_only=True)/converter

In [41]:
for pipe in ['Ukraine Gas Transit', 'Yamal (BY,PL)','Nord Stream', 'Turkstream']:
    entsog_m[pipe] = entsog[entsog['aggregation2'] == pipe]['values'].groupby(pd.Grouper(freq='M')).sum(numeric_only=True)/converter

In [42]:
entsog_q = entsog_m.groupby(pd.Grouper(freq='Q')).sum(numeric_only=True)
lng_q = lng.groupby(pd.Grouper(freq='Q')).sum(numeric_only=True)

In [43]:
graph = entsog_q
graph['USA LNG'] = lng_q['United States']
graph['Russia LNG'] = lng_q['Russia']
del graph['Russia']
graph['LNG less RU and USA'] = lng_q['Total less Russia and USA']
graph = graph['2021':]

In [44]:
graph.tail()

,Norway,Algeria,UK,Azerbaijan,Libya,Ukraine Gas Transit,"Yamal (BY,PL)",Nord Stream,Turkstream,USA LNG,Russia LNG,LNG less RU and USA
dates,,,,,,,,,,,,
2022-12-31,23426.364254,8730.887417,6400.859889,3307.228577,873.443851,3694.221499,0.0,0.0,3147.891807,12624.027920,4566.856133,17305.222743
2023-03-31,23524.158002,7318.667118,4850.583892,3068.134926,691.303740,2834.258186,0.0,0.0,2637.472494,14101.048245,5036.417725,13170.213642
2023-06-30,22424.073815,8509.097454,6484.323957,3037.083973,722.073983,3267.948682,0.0,0.0,2588.875575,16794.255840,4601.492368,14550.795481
2023-09-30,20593.586580,8922.806611,3562.232558,3050.647476,506.069451,3248.204904,0.0,0.0,4388.088996,14462.264634,3878.911022,12677.183567
2023-12-31,23865.734108,8223.530600,3141.930247,3233.510731,667.767167,4261.385733,0.0,0.0,4071.679780,17030.290121,4171.789277,12172.804097


In [45]:
graph = graph[['Nord Stream','Yamal (BY,PL)','Ukraine Gas Transit','Turkstream','Russia LNG', 'USA LNG','LNG less RU and USA', 'Norway','Algeria','UK','Azerbaijan','Libya']]

In [46]:
from datetime import datetime
today = date.today()

In [47]:
with pd.ExcelWriter("Other\quarterly_data {}.xlsx".format(today)) as writer:
    Excelwriter = pd.ExcelWriter("Other\quarterly_data {}.xlsx".format(today),engine="xlsxwriter")
    entsog_q.to_excel(Excelwriter, sheet_name="ENTSOG", index=True)
    lng_q.to_excel(Excelwriter, sheet_name="LNG", index=True)
    graph.to_excel(Excelwriter, sheet_name="graph", index=True)
Excelwriter.close()
Excelwriter.save()

C:\Users\ugne.keliauskaite\AppData\Local\Temp\ipykernel_35792\2207928906.py:7: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  Excelwriter.save()
c:\Users\ugne.keliauskaite\AppData\Local\anaconda3\Lib\site-packages\xlsxwriter\workbook.py:368: UserWarning: Calling close() on already closed file.
  warn("Calling close() on already closed file.")
